<h1>Lab #1 - Object Detection</h1>

**Quiz**

Bitte bearbeiten Sie folgendes Quiz, bevor Sie mit der Bearbeitung beginnen. Mit diesem können Sie sich Zusatzpunkte für die Klausur erarbeiten. Der Laborversuch wird nicht bewertet.

<img src="images/QUIZ_2026-02_KI_LAB_001.svg" width="200px"></img><br>
https://forms.gle/1QhDCgQitCqd7fPA7

**Vorbereitung**

Diese Datei ist ein Jupyter Notebook. Dieses besteht aus Textblöcken im Markdown Format und ausführbaren Code-Zellen. Diese erkennen Sie an dem kleinen Pfeilsymbol links daneben.

Führen Sie bitte als erstes folgende Zelle aus, um sicherzustellen, dass die benötigten Bibliotheken installiert sind:

In [ ]:
!pip install -r requirements.txt

## 1. Motivation

Die automatische Objekterkennung ist eine fundamentale Aufgabe im Bereich Computer Vision und bildet die Grundlage zahlreicher praktischer Anwendungen – von der industriellen Qualitätskontrolle über autonome Robotersysteme bis hin zur Mensch-Maschine-Interaktion. Während klassische Bildverarbeitungsverfahren oft an ihre Grenzen stoßen, sobald Variabilität in Beleuchtung, Perspektive oder Objekterscheinung auftritt, ermöglichen moderne Machine Learning-Ansätze robuste und generalisierbare Lösungen.

In der industriellen Praxis ist es nicht ausreichend, lediglich zu detektieren, *dass* ein Objekt vorhanden ist. Vielmehr müssen präzise Informationen über dessen **Lage, Orientierung und Klasse** extrahiert werden, um beispielsweise einen Robotergreifer korrekt zu positionieren oder Bauteile zu sortieren. Die Herausforderung besteht darin, diese Informationen zuverlässig aus Bilddaten zu gewinnen – selbst unter variierenden Bedingungen.

Dieser Laborversuch vermittelt den vollständigen Workflow einer Machine Learning-basierten Objekterkennung: von der Datenerhebung über das Training neuronaler Netze bis zur Evaluierung und Optimierung. Dabei wird nicht nur theoretisches Wissen vermittelt, sondern Sie erhalten praktische Erfahrung mit den Werkzeugen und Methoden, die in realen Computer Vision-Projekten zum Einsatz kommen.

## 2. Ziel

Das Ziel dieses Laborversuchs ist die Entwicklung eines Machine Learning-Systems zur **Multi-Objekt-Detektion mit Pose-Estimation**. Konkret sollen Sie folgende Kompetenzen erwerben:

**Fachliche Ziele:**
- Entwicklung eines Detektionssystems, das **mehrere Objekte unterschiedlicher Klassen** in einem Bild identifiziert
- Implementierung einer **Keypoint-basierten Pose-Estimation**, wobei jedes Objekt durch **3 charakteristische Keypoints** beschrieben wird, die Lage und Orientierung eindeutig definieren
- Berechnung eines **Objectness-Scores** (Konfidenzwert), der die Sicherheit der Detektion quantifiziert
- Realisierung einer robusten **Multi-Class-Klassifikation**

**Methodische Ziele:**
Sie durchlaufen den **vollständigen Machine Learning Workflow**:
1. **Datenakquise**: Aufnahme und Zusammenstellung eines geeigneten Datensatzes
2. **Datenannotation**: Manuelle Kennzeichnung von Objekten, Klassen und Keypoints
3. **Data Augmentation**: Synthetische Erweiterung des Datensatzes zur Verbesserung der Generalisierung
4. **Data Loading**: Implementierung effizienter Datenladeprozesse für das Training
5. **Model Training**: Training eines neuronalen Netzes auf dem annotierten Datensatz
6. **Evaluierung**: Quantitative Bewertung der Modellleistung mit geeigneten Metriken
7. **Fine Tuning**: Iterative Optimierung von Hyperparametern und Netzarchitektur

Am Ende des Versuchs verfügen Sie über ein funktionsfähiges Objekterkennungssystem und haben praktische Erfahrung mit allen Phasen eines ML-Projekts gesammelt.

## 3. Definition der Datenstruktur

Es soll die Lage und Orientierung von Objekten anhand von 3 Keypoints eindeutig beschrieben werden. Jeder Keypoint ist eine Koordinate im Bild (x, y), wobei sich das Koordinatensystem im Bild oben links befindet:<br>
<img src="images/image_coordinates.png" width="400px"></img>

Die Koordinaten werden als Relativkoordinaten angegeben, um unabhängig von der Bildskalierung zu sein. Befindet sich Keypoint $P_1$ bspw. an Pixelposition $(244, 312)$ in einem Bild mit einer Höhe von 600 Pixeln und einer Breite von 800 Pixeln, so ist die entsprechende Relativkoordinate
$$
    P_1=\begin{pmatrix}
        244px/800px \\
        312px/600px
    \end{pmatrix}=
    \begin{pmatrix}
        0.305 \\
        0.520
    \end{pmatrix}.
$$

Folgende vier Objekttypen sollen im Bild erkannt und klassifiziert werden:

| (0) Bootle Opener - Inlay          | (1) Bootle Opener - Cover           | (2) Bottle Opener             | (3) Cuboid             |
|------------------------------------|-------------------------------------|-------------------------------|------------------------|
| <img src="images/bottle_opener_inlay.png" style="height: 200px; width:auto;"></img> | <img src="images/bottle_opener_cover.png" style="height: 200px; width:auto;"></img> | <img src="images/bottle_opener.png" style="height: 200px; width:auto;"></img> | <img src="images/cuboid.png" style="height: 200px; width:auto;"></img> |
| <img src="images/shapes/Bottle Opener - Inlay.png" style="height: 200px; width:auto;"></img> | <img src="images/shapes/Bottle Opener - Cover.png" style="height: 200px; width:auto;"></img> | <img src="images/shapes/Bottle Opener.png" style="height: 200px; width:auto;"></img> | <img src="images/shapes/Cuboid.png" style="height: 200px; width:auto;"></img> |

_Tabelle 3.1 Objekte_

Die Bilddaten liegen als JPG-Dateien vor. Für jede JPG-Datei wird im selben Verzeichnis eine gleichnamige JSON-Datei erstellt, welche die Annotationen beinhaltet. Jede JSON-Datei beinhaltet eine Liste `[...]` der Objekte im Bild. Jedes Objekt wird durch ein Dictionary `{...}` beschrieben, welches die Attribute `"points"` und `"class"` besitzt. `"points"` ist eine Liste von genau drei Punkten. Das sind die Keypoints in Relativkoordinaten. `"class"` ist ein einzelner `int` Wert, der die Objektklasse (siehe _Tabelle 3.1_) definiert.

**Beispiel:**
```
P1040610.JPG
P1040610.json
    # list of object annotations
    [
        # object dictionary
        {
            "points": [
                [0.711, 0.260],
                [0.583, 0.045],
                [0.493, 0.265]
            ],
            "class": 0
        },
        ...
    ]
```

Weiterhin ist das Datensatzverzeichnis in die Unterverzeichnisse `train`, `val` und `test` unterteilt, wobei alle Daten aus `train` für das Training, alle aus `val` für die Validierung und alle Daten aus `test` für das Testen des Modell verwendet werden.
```
root/
├── train/
│   ├── P1040610.JPG
│   ├── P1040610.json
│   ...
├── val/
│   ├── P1061263.JPG
│   ├── P1061263.json
│   ...
└── test/
    ├── P7835647.JPG
    ├── P7835647.json
    ...
```
Für das Training und die Validierung bekommen Sie bereits annotierte Daten. Ihre Aufgabe wird es sein, die Daten für den Testsplit aufzunehmen, zu annotieren und bereitzustellen. Dazu später mehr.

## 4. Versuchsdurchführung
### 4.1 Datenakquise

Die Datenakquise startet mit der Aufnahme von Bildaten. Für diesen Versuch werden Ihnen Bilder für das Training und die Validierung zur Verfügung gestellt. Diese können unter folgendem Link eingesehen werden: <span style="color:red">TODO: NextCloud Link</span>

**Aufgabe 1**

Erstellen Sie 20 Bilder nach folgenden Kriterien:
- 1 Objekt sichtbar:
  - 2 Bilder, in denen ein _Bottle Opener - Inlay_ zu sehen ist
  - 2 Bilder, in denen ein _Bottle Opener - Cover_ zu sehen ist
  - 2 Bilder, in denen ein _Bottle Opener_ zu sehen ist
  - 2 Bilder, in denen ein _Cuboid_ zu sehen ist
- 2 Objekte sichtbar:
  - 2 Bilder, in denen zwei _Bottle Opener - Inlay_ zu sehen sind
  - 2 Bilder, in denen zwei _Bottle Opener - Cover_ zu sehen sind
  - 2 Bilder, in denen zwei _Bottle Opener_ zu sehen sind
  - 2 Bilder, in denen zwei _Cuboid_ zu sehen sind
- Mindestens 3 Objekte sichtbar:
  - 1 Bild, in dem _Bottle Opener - Inlay_, _Bottle Opener - Cover_, _Bottle Opener_ und _Cuboid_ zu sehen sind
  - 1 Bild, in dem 3 _Bottle Opener - Inlay_ zu sehen sind
  - 1 Bild, in dem 3 _Bottle Opener - Cover_ zu sehen sind
  - 1 Bild, in dem 3 _Bottle Opener_ zu sehen sind
- Variieren Sie die Perspektive zwischen den einzelnen Aufnahmen (Orientierung, Distanz)
- Verwenden Sie möglichst viele unterschiedliche Objekte (bspw. _Bottle Opener - Cover_ mit unterschiedlichen Farben)

**Aufgabe 2**

Führen Sie folgende Codezelle aus, um die grundlegende Verzeichnisstruktur anzulegen. Sie können für `DATASET_DIRECTORY` ein beliebiges Verzeichnis angeben. Wenn Sie diesen Wert auf `"data"` belassen, wird hier ein entsprechendes Unterverzeichnis erstellt.

In [2]:
from pathlib import Path

DATASET_DIRECTORY = 'data'

data_dir = Path(DATASET_DIRECTORY)
for split in ["train", "val", "test"]:
    (data_dir / split).mkdir(parents=True, exist_ok=True)

Stellen Sie nun die von Ihnen aufgenommenen Bilder im Verzeichnis `[DATASET_DIRECTORY]/test` zur Verfügung.

### 4.2 Datan Annotation

Ihnen wird hier ein spezielles OpenCV-basiertes Tool zur Verfügung gestellt, mit dem Sie die aufgenommenen Daten annotieren können.

**Aufgabe 3**

Führen Sie folgende Codezelle aus, um mit der Annotation zu beginnen:

In [ ]:
import src.annotate as ann

ann.SKIP_ANNOTATED = True
ann.annotate_dataset(data_dir)

KeyboardInterrupt: 

: 